In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec


# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils
from analysis_village.cc1pi.BDTs import BDTTrainingUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

import uproot
import awkward as ak
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.colors import to_rgba
from pathlib import Path

In [ ]:
use_range_momentum = True          # base_P = range_P if True else true_P
hits_to_plot = "percent"           # "used", "removed", "percent"
 
hit_bins_used_removed = [0, 50, 100, 200, 300, 500]
hit_bins_percent = [0, 0.25, 0.5, 0.65, 0.8, 0.9]
 
DATA_FILE = "13_subtracted_TLE_5e18_muon_data_update_calo"
MC_FILE   = "13_subtracted_TLE_1e20_muon_update_calo"

# Faint "systematic variation" files: name -> (legend label, line style)
VARIATION_FILES = {
    "13_subtracted_TLE_1e20_muon_alpha_m":   ("alpha -",  ":"),
    "13_subtracted_TLE_1e20_muon_alpha_p":   ("alpha +",  "-."),
    "13_subtracted_TLE_1e20_muon_ccal_m":    ("ccal -",   (0, (5, 2, 1, 2))),
    "13_subtracted_TLE_1e20_muon_ccal_p":    ("ccal +",   (0, (5, 2, 1, 2))),
    "13_subtracted_TLE_wiremod_YZ_muon":     ("wiremod YZ", (0, (1, 1))),
}
VARIATION_ALPHA = 0.35           # faintness of the variation lines
VARIATION_PRESET_KIND = "og"     # which convolution preset the variations use: "og" or "shifted"
show_variations_in_diff = False  # also draw (variation - data) faintly in the lower panel

file_style_labels = ["Data (shifted)", "MC (og)"]


input_dir  = Path("/exp/sbnd/data/users/lpelegri/TLE_processed_files")
output_dir = Path("/exp/sbnd/data/users/lpelegri/TLEGraphs/MoneyPlotComp")

In [ ]:

# momentum binning -> LocalBinInformation("momentum", 16, 0, 0.8) in the macro
n_p_bins = 21
p_low, p_high = 0.2, 0.8
p_bin_edges = np.linspace(p_low, p_high, n_p_bins + 1)
p_bin_centers = 0.5 * (p_bin_edges[:-1] + p_bin_edges[1:])
p_bin_halfwidths = 0.5 * np.diff(p_bin_edges)
 
REF_LABEL = "Range" if use_range_momentum else "True"
P_STRING = rf"$P_{{{REF_LABEL}}}$ [GeV]"
 
SENTINEL = -1000.0  # marks an (hit_bin, p_bin) with zero entries, like the C++ code
 
# qualitative, colorblind-friendly palette -- one color per hit bin
_palette = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7",
    "#E69F00", "#56B4E9", "#999999", "#F0E442",
]
 
hit_bins = hit_bins_percent if hits_to_plot == "percent" else hit_bins_used_removed
n_hit_bins = len(hit_bins)
hit_colors = [_palette[i % len(_palette)] for i in range(n_hit_bins)]

def hit_bin_label(i_hb):
    """Mirrors the legend-string logic of the C++ macro."""
    if hits_to_plot == "percent":
        if i_hb == n_hit_bins - 1:
            return f"Track fraction: > {hit_bins[i_hb]:.2f}"
        return f"Track fraction: {hit_bins[i_hb]:.2f} - {hit_bins[i_hb + 1]:.2f}"
    if i_hb == n_hit_bns - 1:
        return f"{prefix}> {hit_bins[i_hb]}"
    return f"{prefix}{hit_bins[i_hb]} - {hit_bins[i_hb + 1]}"
 
 
def get_hit_bin(values):
    """Vectorised equivalent of GetHitBin(): index of the bin each value falls
    into, right-open, with overflow clamped to the last bin (matching the
    C++ `else return hit_bins.size()-1` overflow behaviour)."""
    edges = np.asarray(hit_bins, dtype=float)
    idx = np.searchsorted(edges[1:], values, side="right")
    return np.minimum(idx, n_hit_bins - 1)

In [ ]:

 
 
# ======================================================================
# 2. READING / FLATTENING THE TREES  (adapted from the uproot example)
# ======================================================================
 
scalar_fields = [
    "true_P", "range_P", "length", "purity", "completeness", "PDG",
    "end_process_string", "out_best_plane", "used_plane",
    "num_used_hits", "num_initial_hits", "num_removed_hits",
]
vector_fields = ["method_vector", "P_track_extension_method", "exit_code"]
 
 
def load_tree_df(file_path):
    """Read one ROOT file and return a flat (event x method) DataFrame with
    all C++-macro-equivalent selections and derived columns already applied."""
    tree = uproot.open(file_path)["tree"]
    arrays = tree.arrays(scalar_fields + vector_fields, library="ak")
    n_events = len(arrays)
    print(f"[{file_path.name}] Entries: {n_events}")
 
    assert ak.all(ak.num(arrays["method_vector"]) == ak.num(arrays["exit_code"]))
    assert ak.all(
        ak.num(arrays["method_vector"]) == ak.num(arrays["P_track_extension_method"])
    )
 
    def to_flat_numpy(ak_array):
        try:
            return ak.to_numpy(ak_array)
        except Exception:
            return np.array(ak.to_list(ak_array), dtype=object)
 
    counts = ak.to_numpy(ak.num(arrays["method_vector"], axis=1))
    data = {"entry": np.repeat(np.arange(n_events), counts)}
    for f in scalar_fields:
        data[f] = np.repeat(to_flat_numpy(arrays[f]), counts)
    for f in vector_fields:
        data[f] = to_flat_numpy(ak.flatten(arrays[f], axis=1))
 
    df = pd.DataFrame(data).rename(columns={
        "PDG": "pdg",
        "method_vector": "method",
        "P_track_extension_method": "reco_P_method",
    })
 
    # -- selection identical to the C++ loop --------------------------------
    df = df[df["exit_code"] == 0].copy()
 
    df["base_P"] = df["range_P"] if use_range_momentum else df["true_P"]

    df["residual"] = (df["reco_P_method"] - df["base_P"]) / df["base_P"]
    df = df[np.isfinite(df["residual"])]
    df = df[(df["residual"] > -1.5) & (df["residual"] < 1.5)].copy()
 
    df["p_type_specific"] = np.where(
        df["end_process_string"].isin(["pipInelastic", "pimInelastic"]),
        "Inelastic",
        "Stopping",
    )
 
    if hits_to_plot == "removed":
        df["hits_metric"] = df["num_removed_hits"]
    elif hits_to_plot == "percent":
        df["hits_metric"] = df["num_used_hits"] / df["num_initial_hits"]
    else:  # "used"
        df["hits_metric"] = df["num_used_hits"]
 
    df["hit_bin"] = get_hit_bin(df["hits_metric"].values)
    df["p_bin"] = np.digitize(df["base_P"].values, p_bin_edges) - 1
    df = df[(df["p_bin"] >= 0) & (df["p_bin"] < n_p_bins)].copy()
 
    return df.reset_index(drop=True)
 
df_data = load_tree_df(input_dir / f"{DATA_FILE}.root")
df_mc   = load_tree_df(input_dir / f"{MC_FILE}.root")
df_vars = {name: load_tree_df(input_dir / f"{name}.root") for name in VARIATION_FILES}

presets = sorted(set(df_data["method"].unique()) | set(df_mc["method"].unique()))
print("Presets found:", presets)

shifted_preset = next(p for p in presets if "convolution" in p and "shifted" in p)
og_preset      = next(p for p in presets if "convolution" in p and "og" in p)
var_preset     = og_preset if VARIATION_PRESET_KIND == "og" else shifted_preset
print(f"Data preset: {shifted_preset}   MC preset: {og_preset}   Variation preset: {var_preset}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.interpolate import PchipInterpolator

SHOW_VARIATION_BANDS = True
STAT_ALPHA = 0.35      # MC stat band
SYST_ALPHA = 0.15      # MC stat + variations band
SMOOTH_BANDS = True   # smooth curve through bin centers (False = straight segments)

# Variation (systematic) error shape
SYMMETRIC_SYST = False # True: use the bigger of (up, down) on both sides; False: asymmetric up/down

# How variations are combined into the MC systematic:
#   "quadrature": one shift per source (SYST_GROUPS), sources added in quadrature
#   "envelope"  : all variations treated as one group -> outer band (old behaviour)
SYST_MODE = "envelope"
SUBTRACT_VAR_STAT = True

# source name -> variation files belonging to it (keys of VARIATION_FILES).
# Files that are not listed are treated as their own source.
SYST_GROUPS = {
    "alpha":   ["13_subtracted_TLE_1e20_muon_alpha_m", "13_subtracted_TLE_1e20_muon_alpha_p"],
    "ccal":    ["13_subtracted_TLE_1e20_muon_ccal_m","13_subtracted_TLE_1e20_muon_ccal_p"],
    "wiremod": ["13_subtracted_TLE_wiremod_YZ_muon"],
}

# Variation files whose events differ from the CV sample -> stat noise of (var - nom) is removed.
# All other variations are assumed to be the same events as the CV -> no subtraction.
INDEPENDENT_VARIATIONS = {"13_subtracted_TLE_wiremod_YZ_muon"}

# Position of the top-panel legend, per statistic key of stat_defs
LEGEND_LOC = {"mean": "lower left", "std": "upper right"}

# All text sizes (labels, titles, ticks, legends, text box) are shifted by this many points
FONT_SHIFT = 2

# Bottom panel layout
FIG_SIZE = (8, 9)
HEIGHT_RATIOS = [6, 4]
DIFF_YLIM = (-0.2, 0.2)
DODGE_FRACTION = 0.9   # fraction of the p-bin width shared by all hit bins (points are shifted sideways)

def compute_profile(df, preset, clip_range=None, min_n_eff=1, verbose=False):
    """Returns (mean, mean_err, std, std_err), each shaped
    [n_hit_bins, n_p_bins].

    mean/std are computed from all rows in each (hit_bin, p_bin) group.
    mean_err/std_err use n_eff = number of *unique tracks* in that group,
    where a track is identified by (true_P, num_initial_hits), since a
    single track can appear multiple times (e.g. across methods) and
    using the raw row count would double-count statistics.

    clip_range: if set, only residuals within [-clip_range, clip_range]
    are included (both for mean/std and for counting n_eff).

    verbose: if True, print hit_bin, p_bin, group size before the clip,
    group size after the clip, and n_eff (unique tracks) after.
    """
    sub = df[df["method"] == preset]

    mean = np.full((n_hit_bins, n_p_bins), SENTINEL)
    mean_err = np.zeros((n_hit_bins, n_p_bins))
    std = np.full((n_hit_bins, n_p_bins), SENTINEL)
    std_err = np.zeros((n_hit_bins, n_p_bins))

    if verbose:
        print(f"--- compute_profile: preset = {preset} ---")

    track_id_cols = ["true_P", "num_initial_hits"]
    cols = ["residual"] + track_id_cols
    for (i_hb, i_pb), g in sub.groupby(["hit_bin", "p_bin"])[cols]:
        vals = g["residual"]
        n_before = len(vals)

        if clip_range is not None:
            keep = vals.abs() <= clip_range
            g = g.loc[keep]
            vals = vals[keep]

        n = len(vals)
        if n == 0:
            if verbose:
                print(f"  hit_bin={i_hb} p_bin={i_pb}: n_before={n_before}, "
                      f"n_after_clip=0, n_eff=0 (skipped)")
            continue

        n_eff = g[track_id_cols].drop_duplicates().shape[0]

        if verbose:
            print(f"  hit_bin={i_hb} p_bin={i_pb}: n_before={n_before}, "
                  f"n_after_clip={n}, n_eff (unique tracks)={n_eff}")

        s = vals.std(ddof=0)
        mean[i_hb, i_pb] = vals.mean()
        std[i_hb, i_pb] = s

        if n_eff < min_n_eff:
            mean_err[i_hb, i_pb] = np.nan
            std_err[i_hb, i_pb] = np.nan
            if verbose:
                print(f"    -> n_eff below min_n_eff={min_n_eff}, errors set to NaN")
        else:
            mean_err[i_hb, i_pb] = s / np.sqrt(n_eff)
            std_err[i_hb, i_pb] = s / np.sqrt(2 * n_eff)

    return mean, mean_err, std, std_err

def mc_uncertainty(variations, nominal, nominal_err):
    """variations: dict {file name: [n_hit_bins, n_p_bins] array of the same statistic}.
    Returns dict of [n_hit_bins, n_p_bins] arrays (NaN where nominal is empty):
    nom, e_mc, s_up, s_dn  (s_* = systematic from the variations only).

    Per source (group of files), with d = variation - nominal:
        up = max(0, max d),  dn = max(0, -min d)
        if SYMMETRIC_SYST: up = dn = max(up, dn)
    SYST_MODE "quadrature": s_up = sqrt(sum_sources up^2), same for s_dn.
    SYST_MODE "envelope"  : all files form one source (outer band)."""
    valid = nominal != SENTINEL
    nom = np.where(valid, nominal, np.nan)
    e_mc = np.where(valid, nominal_err, np.nan)

    if SYST_MODE == "envelope":
        groups = {"all": list(variations)}
    elif SYST_MODE == "quadrature":
        groups = {g: [n for n in names if n in variations] for g, names in SYST_GROUPS.items()}
        listed = {n for names in SYST_GROUPS.values() for n in names}
        groups.update({n: [n] for n in variations if n not in listed})
    else:
        raise ValueError(f"Unknown SYST_MODE: {SYST_MODE!r}")

    up2 = np.zeros_like(nom)
    dn2 = np.zeros_like(nom)
    for names in groups.values():
        if not names:
            continue
        dv = np.stack([np.where(variations[n] != SENTINEL, variations[n] - nom, 0.0)
                       for n in names])
        up = np.maximum(dv.max(axis=0), 0.0)
        dn = np.maximum(-dv.min(axis=0), 0.0)
        if SYMMETRIC_SYST:
            up = dn = np.maximum(up, dn)
        up2 = up2 + up ** 2
        dn2 = dn2 + dn ** 2
    return dict(nom=nom, e_mc=e_mc, s_up=np.sqrt(up2), s_dn=np.sqrt(dn2))


def compute_rmsd(v0, v1, sentinel=SENTINEL):
    """RMSD of (v0 - v1) over all valid (hit_bin, p_bin) cells."""
    mask = (v0 != sentinel) & (v1 != sentinel)
    if not np.any(mask):
        return None
    return float(np.sqrt(np.mean((v0[mask] - v1[mask]) ** 2)))


# ======================================================================
# 4. PLOTTING
# ======================================================================

def draw_band(ax, lo, hi, alpha, zorder):
    """Band through the bin centers, one per hit bin; runs of consecutive
    valid bins are drawn separately. Each run is extended flat out to the
    outer edge of its first and last bin."""
    for i_hb in range(n_hit_bins):
        idx = np.flatnonzero(np.isfinite(lo[i_hb]) & np.isfinite(hi[i_hb]))
        if len(idx) == 0:
            continue
        for run in np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1):
            x, l, h = p_bin_centers[run], lo[i_hb, run], hi[i_hb, run]
            if SMOOTH_BANDS and len(run) >= 3:
                xs = np.linspace(x[0], x[-1], 20 * len(x))
                ls, hs = PchipInterpolator(x, l)(xs), PchipInterpolator(x, h)(xs)
                ls, hs = np.minimum(ls, hs), np.maximum(ls, hs)
            else:
                xs, ls, hs = x, l, h
            # flat extension to the edges of the first / last bin
            xs = np.concatenate([[p_bin_edges[run[0]]], xs, [p_bin_edges[run[-1] + 1]]])
            ls = np.concatenate([[ls[0]], ls, [ls[-1]]])
            hs = np.concatenate([[hs[0]], hs, [hs[-1]]])
            ax.fill_between(xs, ls, hs, color=hit_colors[i_hb],
                            alpha=alpha, linewidth=0, zorder=zorder)


def draw_uncertainty_bands(ax, u):
    nom, e = u["nom"], u["e_mc"]
    draw_band(ax, nom - np.hypot(e, u["s_dn"]), nom + np.hypot(e, u["s_up"]),
              SYST_ALPHA, zorder=1)                        # stat + variations (light)
    draw_band(ax, nom - e, nom + e, STAT_ALPHA, zorder=2)  # MC stat only (darker)


def plot_profile_overlay(values, errs, ax, y_range, y_label, title,
                         show_xlabel=True, unc=None):
    if unc is not None:
        draw_uncertainty_bands(ax, unc)
    for i_hb in range(n_hit_bins):
        for i_f in range(2):
            v, e = values[i_f][i_hb], errs[i_f][i_hb]
            mask = v != SENTINEL
            marker = "o" if i_f == 0 else "s"
            linestyle = "-" if i_f == 0 else "--"
            ax.errorbar(
                p_bin_centers[mask], v[mask], yerr=e[mask],
                xerr=p_bin_halfwidths[mask], marker=marker, linestyle=linestyle,
                color=hit_colors[i_hb], markersize=5, capsize=2, linewidth=1.2,
                zorder=4,
            )
    ax.axhline(0, color="gray", linestyle="--", linewidth=1.5, zorder=0)
    if show_xlabel:
        ax.set_xlabel(P_STRING, fontsize=13 + FONT_SHIFT)
    ax.set_ylabel(y_label, fontsize=13 + FONT_SHIFT)
    ax.set_ylim(*y_range)
    ax.set_title(title, fontsize=13 + FONT_SHIFT)
    ax.grid(alpha=0.25)
    ax.tick_params(labelsize=10 + FONT_SHIFT)


def plot_difference_overlay(values, errs, ax, y_label, title=None,
                            show_xlabel=True, unc=None):
    """Bottom panel: Data - MC per hit bin.

    - Hit bins are dodged sideways inside each p bin (alternate p bins are
      striped so it is clear which bin a point belongs to).
    - Nested error bars (same color):
        thick   : Data stat only
        medium  : Data stat (+) MC stat
        thin+cap: Data stat (+) MC stat (+) MC variations (asymmetric unless SYMMETRIC_SYST)
    - Marker filled  -> point is within the total uncertainty (zero inside the bar)
      Marker open    -> point is outside the total uncertainty

    Returns (n_in, n_out): number of points with zero inside / outside
    the total uncertainty.
    """
    step = DODGE_FRACTION * (p_bin_edges[1] - p_bin_edges[0]) / n_hit_bins
    n_in, n_out = 0, 0

    for j in range(0, n_p_bins, 2):
        ax.axvspan(p_bin_edges[j], p_bin_edges[j + 1], color="gray",
                   alpha=0.08, linewidth=0, zorder=0)

    for i_hb in range(n_hit_bins):
        v0, v1 = values[0][i_hb], values[1][i_hb]
        e0, e1 = errs[0][i_hb], errs[1][i_hb]
        mask = (v0 != SENTINEL) & (v1 != SENTINEL)
        if not np.any(mask):
            continue
        color = hit_colors[i_hb]
        diff = v0[mask] - v1[mask]
        e_data = e0[mask]
        e_stat = np.sqrt(e0[mask] ** 2 + e1[mask] ** 2)      # Data (+) MC stat
        x = p_bin_centers[mask] + (i_hb - (n_hit_bins - 1) / 2) * step

        if unc is not None:
            # MC up by s_up  ->  (Data - MC) down by s_up, and vice versa
            err_lo = np.sqrt(e_stat ** 2 + unc["s_up"][i_hb][mask] ** 2)
            err_hi = np.sqrt(e_stat ** 2 + unc["s_dn"][i_hb][mask] ** 2)
            ax.errorbar(x, diff, yerr=[err_lo, err_hi], fmt="none", ecolor=color,
                        elinewidth=0.8, capsize=1.5, capthick=0.8, zorder=2)
        else:
            err_lo = err_hi = e_stat

        ax.errorbar(x, diff, yerr=e_stat, fmt="none", ecolor=color,
                    elinewidth=1.5, capsize=0, zorder=3)
        ax.errorbar(x, diff, yerr=e_data, fmt="none", ecolor=color,
                    elinewidth=2.8, capsize=0, zorder=4)

        consistent = (diff - err_lo <= 0) & (diff + err_hi >= 0)
        n_in += int(consistent.sum())
        n_out += int((~consistent).sum())
        ax.scatter(x, diff, s=14, marker="o", zorder=5, linewidths=1.0,
                   edgecolors=color,
                   facecolors=[color if c else "white" for c in consistent])

    ax.axhline(0, color="gray", linestyle="--", linewidth=1.5, zorder=0)
    if show_xlabel:
        ax.set_xlabel(P_STRING, fontsize=13 + FONT_SHIFT)
    ax.set_ylabel(y_label, fontsize=13 + FONT_SHIFT)
    ax.set_ylim(*DIFF_YLIM)
    if title:
        ax.set_title(title, fontsize=13 + FONT_SHIFT)
    ax.grid(alpha=0.25)
    ax.tick_params(labelsize=10 + FONT_SHIFT)
    return n_in, n_out


def diff_legend_handles(with_variations):
    h = [
        Line2D([0], [0], color="black", lw=2.8, label="Data stat"),
        Line2D([0], [0], color="black", lw=1.5, label=r"$\oplus$ MC stat"),
    ]
    if with_variations:
        h.append(Line2D([0], [0], color="black", lw=0.8, marker="|", markersize=5,
                        label=r"$\oplus$ MC variations"))
    h += [
        Line2D([0], [0], color="black", marker="o", linestyle="none",
               markerfacecolor="black", markersize=4, label="within total unc."),
        Line2D([0], [0], color="black", marker="o", linestyle="none",
               markerfacecolor="white", markersize=4, label="outside total unc."),
    ]
    return h


def plot_combined(values, errs, y_range, overlay_label, diff_label, title,
                  legend_handles, rmsd=None, variations=None, legend_loc="lower left"):
    """One figure: overlay panel on top, difference panel below, sharing the x-axis.
    Returns (fig, n_in, n_out)."""
    fig = plt.figure(figsize=FIG_SIZE)
    gs = GridSpec(2, 1, height_ratios=HEIGHT_RATIOS, hspace=0.08, figure=fig)
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    unc = None
    if variations and SHOW_VARIATION_BANDS:
        unc = mc_uncertainty(variations, nominal=values[1], nominal_err=errs[1])

    plot_profile_overlay(values, errs, ax_top, y_range, overlay_label, title,
                         show_xlabel=False, unc=unc)
    plt.setp(ax_top.get_xticklabels(), visible=False)

    n_in, n_out = plot_difference_overlay(values, errs, ax_bot, diff_label,
                                          title=None, show_xlabel=True, unc=unc)

    ax_top.legend(handles=legend_handles, fontsize=8 + FONT_SHIFT, loc=legend_loc)
    ax_bot.legend(handles=diff_legend_handles(unc is not None), fontsize=7 + FONT_SHIFT,
                  loc="upper left", ncol=3, framealpha=0.85)

    n_tot = n_in + n_out
    if n_tot:
        lines = [f"within unc.: {n_in}/{n_tot} ({n_in / n_tot:.0%})",
                 f"outside unc.: {n_out}/{n_tot}"]
    else:
        lines = ["no points"]
    if rmsd is not None:
        lines.append(f"RMSD = {rmsd:.4f}")
    ax_bot.text(
        0.97, 0.05, "\n".join(lines), transform=ax_bot.transAxes, fontsize=8 + FONT_SHIFT,
        verticalalignment="bottom", horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8, edgecolor="gray"),
    )
    return fig, n_in, n_out


def make_legend(label_0, label_1):
    handles = [
        Line2D([0], [0], color=hit_colors[i], lw=2, label=hit_bin_label(i))
        for i in range(n_hit_bins)
    ]
    handles += [
        Line2D([0], [0], color="black", marker="o", linestyle="-", label=label_0),
        Line2D([0], [0], color="black", marker="s", linestyle="--", label=label_1),
    ]
    if SHOW_VARIATION_BANDS:
        handles.append(Patch(facecolor="gray", alpha=STAT_ALPHA + 0.1, label="MC stat"))
        handles.append(Patch(facecolor="gray", alpha=SYST_ALPHA + 0.1,
                             label="MC stat ⊕ variations"))
    return handles


stat_defs = [
    # folder, stat key, overlay y-range, overlay label, diff label
    (
        "Bias", "mean", (-1, 0.2),
        rf"$\mu\left(\frac{{P_{{Reco}}-P_{{{REF_LABEL}}}}}{{P_{{{REF_LABEL}}}}}\right)$",
        r"$\mu_{Data} - \mu_{MC}$",
    ),
    (
        "StdDev", "std", (0, 0.5),
        rf"$\sigma\left(\frac{{P_{{Reco}}-P_{{{REF_LABEL}}}}}{{P_{{{REF_LABEL}}}}}\right)$",
        r"$\sigma_{Data} - \sigma_{MC}$",
    ),
]


# ======================================================================
# 5. MAIN LOOP
# ======================================================================

legend = make_legend("Data", "MC")

for preset in presets:
    data_profile = compute_profile(df_data, preset, clip_range=1.5)
    mc_profile = compute_profile(df_mc, preset, clip_range=1.5)
    var_profiles_p = {
        name: compute_profile(df, preset, clip_range=1.5)
        for name, df in df_vars.items()
    }

    # index 0 = data, index 1 = MC
    mean_vals = [data_profile[0], mc_profile[0]]
    mean_errs = [data_profile[1], mc_profile[1]]
    std_vals = [data_profile[2], mc_profile[2]]
    std_errs = [data_profile[3], mc_profile[3]]

    for folder, stat, y_range, overlay_label, diff_label in stat_defs:
        values = mean_vals if stat == "mean" else std_vals
        errs = mean_errs if stat == "mean" else std_errs

        k = 0 if stat == "mean" else 2
        variations = {n: var_profiles_p[n][k] for n in VARIATION_FILES}

        rmsd = compute_rmsd(values[0], values[1])

        out_dir = output_dir / folder
        out_dir.mkdir(parents=True, exist_ok=True)

        fig, n_in, n_out = plot_combined(
            values, errs, y_range, overlay_label, diff_label, preset,
            legend, rmsd=rmsd, variations=variations,
            legend_loc=LEGEND_LOC[stat],
        )
        fig.savefig(out_dir / f"{preset}.pdf")
        # plt.close(fig)

        print(f"[{preset} - {stat}] within: {n_in}, outside: {n_out}"
              + (f", RMSD: {rmsd:.4f}" if rmsd is not None else ""))

print(f"Done. Plots written under: {output_dir}")


# ======================================================================
# 6. DATA (shifted) vs MC (og) COMPARISON
#    Kept at the END of the script on purpose: it needs compute_profile,
#    plot_combined, make_legend, stat_defs, presets, df_data, df_mc and
#    df_vars to already exist, so it breaks if moved above them.
# ======================================================================

VARIATION_PRESET_KIND = "og"   # preset used by the variation files: "og" or "shifted"
                               # (keep "og": the bands are built around the MC nominal)

def find_preset(*keywords):
    for p in presets:
        if all(k in p for k in keywords):
            return p
    raise RuntimeError(f"No preset containing {keywords} found in {presets}")

shifted_preset = find_preset("convolution", "shifted")
og_preset = find_preset("convolution", "og")
var_preset = og_preset if VARIATION_PRESET_KIND == "og" else shifted_preset
print(f"Data preset: {shifted_preset}   MC preset: {og_preset}   Variation preset: {var_preset}")

cmp_data_profile = compute_profile(df_data, shifted_preset, clip_range=1.5)
cmp_mc_profile = compute_profile(df_mc, og_preset, clip_range=1.5)
cmp_var_profiles = {
    name: compute_profile(df, var_preset, clip_range=1.5)
    for name, df in df_vars.items()
}

# index 0 = data (solid, circle), index 1 = MC (dashed, square)
mean_vals_cmp = [cmp_data_profile[0], cmp_mc_profile[0]]
mean_errs_cmp = [cmp_data_profile[1], cmp_mc_profile[1]]
std_vals_cmp = [cmp_data_profile[2], cmp_mc_profile[2]]
std_errs_cmp = [cmp_data_profile[3], cmp_mc_profile[3]]


def get_variations(stat):
    k = 0 if stat == "mean" else 2
    return {n: cmp_var_profiles[n][k] for n in VARIATION_FILES}


legend_cmp = make_legend("Data (shifted)", "MC (og)")

for folder, stat, y_range, overlay_label, diff_label in stat_defs:
    values = mean_vals_cmp if stat == "mean" else std_vals_cmp
    errs = mean_errs_cmp if stat == "mean" else std_errs_cmp

    rmsd = compute_rmsd(values[0], values[1])

    out_dir = output_dir / folder
    out_dir.mkdir(parents=True, exist_ok=True)

    fig, n_in, n_out = plot_combined(
        values, errs, y_range, overlay_label, diff_label,
        "Data (shifted) vs MC (og)", legend_cmp,
        rmsd=rmsd, variations=get_variations(stat),
        legend_loc=LEGEND_LOC[stat],
    )
    fig.savefig(out_dir / "data_shifted_vs_mc_og.pdf")
    # plt.close(fig)

    print(f"[Data(shifted) vs MC(og) - {stat}] within: {n_in}, outside: {n_out}"
          + (f", RMSD: {rmsd:.4f}" if rmsd is not None else ""))

print(f"Done. Data-shifted-vs-MC-og comparison written under: {output_dir}")